
# Lateral Tire Model Test & Debug
횡방향 타이어 모델 테스트 및 디버깅 노트북

## 목적
- LateralTireModel의 기본 동작 및 슬립각 계산 확인
- **입력**: φ_st, V_wheel_x, V_wheel_y, F_tire → **출력**: F_y, M_z 응답 검증
- 기본 모델: `F_y = -C_alpha * F_z * α` (|F_y| ≤ μ·F_z), `M_z = -trail * F_y`


In [ ]:

# 필요한 라이브러리 import
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 프로젝트 루트를 Python path에 추가
project_root = Path.cwd().parent.parent.parent.parent.parent.parent
sys.path.insert(0, str(project_root))

# LateralTireModel import
from vehicle_sim.models.e_corner.tire.lateral.lateral_tire import LateralTireModel

print("Import 성공!")



## 1. 기본 동작 테스트
YAML 기본 파라미터 로드 및 초기 상태 확인


In [ ]:

# 모델 생성
lateral_tire = LateralTireModel()

print("=== Lateral Tire Parameters ===")
print(f"C_alpha (cornering stiffness): {lateral_tire.params.C_alpha} [N/rad]")
print(f"mu (friction coefficient): {lateral_tire.params.mu}")
print(f"trail (aligning torque arm): {lateral_tire.params.trail} [m]")

print("=== 초기 상태 ===")
print(lateral_tire.get_state())



## 2. 시계열 응답 테스트 - 정속 주행, 조향 각 변화
**입력**: 일정한 속도(V_x), 스텝/램프 형태의 조향 각 φ_st, 일정한 수직하중 F_tire  
**출력**: 슬립각 α, 횡방향 힘 F_y, 얼라이닝 토크 M_z


In [ ]:

# 시뮬레이션 설정
dt = 0.01
t_end = 6.0
time = np.arange(0, t_end, dt)
steps = len(time)

# 입력 설정
V_wheel_x = 15.0  # [m/s]
F_tire_const = 4000.0  # [N]

phi_history = np.zeros(steps)
Fz_history = np.full(steps, F_tire_const)
alpha_history = np.zeros(steps)
Fy_history = np.zeros(steps)
Mz_history = np.zeros(steps)

# 모델 리셋
lateral_tire.reset()

for i, t in enumerate(time):
    # [입력] 조향각 시나리오: 0~2초 0rad, 2~4초 램프, 4~6초 유지
    if t < 2.0:
        steering_angle = 0.0
    elif t < 4.0:
        steering_angle = np.deg2rad(5.0) * (t - 2.0) / 2.0  # 0 → 5deg 램프
    else:
        steering_angle = np.deg2rad(5.0)

    # 조향각에 따른 횡방향 휠 속도 계산 (휠 프레임 변환)
    V_wheel_y = V_wheel_x * np.tan(steering_angle)

    # 모델 업데이트
    Fy = lateral_tire.update(V_wheel_x, V_wheel_y, F_tire_const)

    # 상태에서 슬립각/얼라이닝 토크 추출
    alpha = lateral_tire.state.slip_angle
    Mz = lateral_tire.state.aligning_torque

    # 기록
    phi_history[i] = steering_angle
    alpha_history[i] = alpha
    Fy_history[i] = Fy
    Mz_history[i] = Mz

print("정속 조향 시나리오 시뮬레이션 완료!")
print(f"최종 슬립각: {np.rad2deg(alpha_history[-1]):.2f} deg")
print(f"최종 횡방향 힘: {Fy_history[-1]:.1f} N, 얼라이닝 토크: {Mz_history[-1]:.1f} N·m")


In [ ]:

# ========================================
# [입력 INPUT] 시각화 - 정속 조향 시나리오
# ========================================
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('[INPUT] Lateral Tire Inputs - Constant Speed Steering', fontsize=16, fontweight='bold', color='blue')

axes[0].plot(time, np.rad2deg(phi_history), 'b-', linewidth=2.5)
axes[0].set_ylabel('Steering Angle φ_st [deg]', fontsize=13, fontweight='bold')
axes[0].set_title('Steering Angle Profile', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, time[-1]])

axes[1].plot(time, Fz_history, 'orange', linewidth=2.5)
axes[1].set_xlabel('Time [s]', fontsize=13)
axes[1].set_ylabel('F_tire [N]', fontsize=13, fontweight='bold')
axes[1].set_title('Normal Load (constant)', fontsize=13)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, time[-1]])

plt.tight_layout()
plt.show()

# ========================================
# [출력 OUTPUT] 시각화 - 정속 조향 시나리오
# ========================================
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('[OUTPUT] Lateral Tire Outputs - Constant Speed Steering', fontsize=16, fontweight='bold', color='green')

axes[0].plot(time, np.rad2deg(alpha_history), 'g-', linewidth=2.5)
axes[0].set_ylabel('Slip Angle α [deg]', fontsize=13, fontweight='bold')
axes[0].set_title('Slip Angle', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim([0, time[-1]])

axes[1].plot(time, Fy_history, 'r-', linewidth=2.5)
axes[1].set_ylabel('Lateral Force F_y [N]', fontsize=13, fontweight='bold')
axes[1].set_title('Lateral Force', fontsize=13)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([0, time[-1]])

axes[2].plot(time, Mz_history, 'purple', linewidth=2.5)
axes[2].set_xlabel('Time [s]', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Aligning Torque M_z [N·m]', fontsize=13, fontweight='bold')
axes[2].set_title('Aligning Torque', fontsize=13)
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim([0, time[-1]])

plt.tight_layout()
plt.show()



## 3. 슬립각-횡력 특성 곡선
정지 상태에서 슬립각 α를 -10° ~ +10° 범위로 변화시키며 F_y, M_z 특성 확인


In [ ]:

# 슬립각 스윕 설정
alpha_deg = np.linspace(-10.0, 10.0, 201)
alpha_rad = np.deg2rad(alpha_deg)

Fz_values = [2000.0, 4000.0, 6000.0]
Fy_map = {}
Mz_map = {}

for Fz in Fz_values:
    Fy_list = []
    Mz_list = []
    for alpha in alpha_rad:
        Fy = lateral_tire.calculate_force(alpha, Fz)
        Mz = lateral_tire.calculate_aligning_torque(alpha, Fz, Fy_override=Fy)
        Fy_list.append(Fy)
        Mz_list.append(Mz)
    Fy_map[Fz] = np.array(Fy_list)
    Mz_map[Fz] = np.array(Mz_list)

print("슬립각 스윕 계산 완료!")


In [ ]:

# 결과 시각화 - F_y vs α, M_z vs α
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Lateral Tire Characteristics vs Slip Angle', fontsize=16, fontweight='bold', color='blue')

colors = ['b', 'g', 'r']
for Fz, color in zip(Fz_values, colors):
    axes[0].plot(alpha_deg, Fy_map[Fz], color=color, linewidth=2.5, label=f'F_tire={Fz:.0f} N')

axes[0].set_ylabel('Lateral Force F_y [N]', fontsize=13, fontweight='bold')
axes[0].set_title('Lateral Force vs Slip Angle', fontsize=13)
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=11)

for Fz, color in zip(Fz_values, colors):
    axes[1].plot(alpha_deg, Mz_map[Fz], color=color, linewidth=2.5, label=f'F_tire={Fz:.0f} N')

axes[1].set_xlabel('Slip Angle α [deg]', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Aligning Torque M_z [N·m]', fontsize=13, fontweight='bold')
axes[1].set_title('Aligning Torque vs Slip Angle', fontsize=13)
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()



## 4. 파라미터 변화 테스트 (C_alpha, μ, trail)
슬립각 입력이 주어졌을 때 파라미터 변화에 따른 F_y, M_z 응답 비교


In [ ]:

# 공통 슬립각 입력 (사인파)
dt = 0.01
t_end = 5.0
time = np.arange(0, t_end, dt)
alpha_input = np.deg2rad(5.0) * np.sin(2 * np.pi * 0.5 * time)  # ±5 deg
Fz_param = 4000.0

# 4-1. C_alpha 변화
C_values = [40000.0, 80000.0, 120000.0]
Fy_C = {}

for C in C_values:
    model = LateralTireModel()
    model.params.C_alpha = C
    Fy_list = []
    for alpha in alpha_input:
        Fy_list.append(model.calculate_force(alpha, Fz_param))
    Fy_C[C] = np.array(Fy_list)

# 4-2. μ 변화
mu_values = [0.6, 1.0, 1.4]
Fy_mu = {}

for mu in mu_values:
    model = LateralTireModel()
    model.params.mu = mu
    Fy_list = []
    for alpha in alpha_input:
        Fy_list.append(model.calculate_force(alpha, Fz_param))
    Fy_mu[mu] = np.array(Fy_list)

# 4-3. trail 변화 (M_z에만 영향)
trail_values = [0.02, 0.05, 0.08]
Mz_trail = {}

for trail in trail_values:
    model = LateralTireModel()
    model.params.trail = trail
    Mz_list = []
    for alpha in alpha_input:
        Fy = model.calculate_force(alpha, Fz_param)
        Mz_list.append(model.calculate_aligning_torque(alpha, Fz_param, Fy_override=Fy))
    Mz_trail[trail] = np.array(Mz_list)

print("파라미터 변화 시뮬레이션 완료!")


In [ ]:

# C_alpha 변화에 따른 F_y 비교
fig, ax = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Parameter Sweep: C_alpha, μ, trail', fontsize=16, fontweight='bold', color='purple')

# (1,1) 입력 슬립각
ax[0, 0].plot(time, np.rad2deg(alpha_input), 'k-', linewidth=2.0)
ax[0, 0].set_ylabel('α [deg]', fontsize=11, fontweight='bold')
ax[0, 0].set_title('Slip Angle Input', fontsize=12)
ax[0, 0].grid(True, alpha=0.3)

# (1,2) C_alpha 변화
colors = ['b', 'g', 'r']
for (C, Fy), color in zip(Fy_C.items(), colors):
    ax[0, 1].plot(time, Fy, color=color, linewidth=2.0, label=f'C_alpha={C:.0f}')
ax[0, 1].set_title('Effect of C_alpha on F_y', fontsize=12)
ax[0, 1].set_ylabel('F_y [N]', fontsize=11, fontweight='bold')
ax[0, 1].grid(True, alpha=0.3)
ax[0, 1].legend(fontsize=9)

# (2,1) μ 변화
for (mu, Fy), color in zip(Fy_mu.items(), colors):
    ax[1, 0].plot(time, Fy, color=color, linewidth=2.0, label=f'mu={mu:.1f}')
ax[1, 0].set_title('Effect of μ on F_y (saturation)', fontsize=12)
ax[1, 0].set_xlabel('Time [s]', fontsize=11, fontweight='bold')
ax[1, 0].set_ylabel('F_y [N]', fontsize=11, fontweight='bold')
ax[1, 0].grid(True, alpha=0.3)
ax[1, 0].legend(fontsize=9)

# (2,2) trail 변화에 따른 M_z
for (trail, Mz), color in zip(Mz_trail.items(), colors):
    ax[1, 1].plot(time, Mz, color=color, linewidth=2.0, label=f'trail={trail:.2f} m')
ax[1, 1].set_title('Effect of trail on M_z', fontsize=12)
ax[1, 1].set_xlabel('Time [s]', fontsize=11, fontweight='bold')
ax[1, 1].set_ylabel('M_z [N·m]', fontsize=11, fontweight='bold')
ax[1, 1].grid(True, alpha=0.3)
ax[1, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()



## 5. Reset 기능 테스트
update 후 reset() 호출 시 상태가 초기화되는지 확인


In [ ]:

reset_test = LateralTireModel()

print("초기 상태:", reset_test.get_state())

# 업데이트 수행 (V_wheel_y = V_wheel_x * tan(5deg) 로 슬립각 생성)
V_wheel_x = 10.0
V_wheel_y = V_wheel_x * np.tan(np.deg2rad(5.0))
Fy_before = reset_test.update(V_wheel_x, V_wheel_y, 3000.0)
print(f"업데이트 후 상태: {reset_test.get_state()} (F_y={Fy_before:.1f} N)")

# Reset 호출
reset_test.reset()
print("Reset 후 상태:", reset_test.get_state())

assert reset_test.state.slip_angle == 0.0 and reset_test.state.lateral_force == 0.0 and reset_test.state.aligning_torque == 0.0, \
    "Reset이 제대로 동작하지 않음!"
print("✓ Reset 기능 정상 동작")



## 6. 요약
- ✓ 기본 파라미터/상태 확인
- ✓ 정속 조향 시나리오에서 α, F_y, M_z 응답 확인
- ✓ 슬립각-횡력/얼라이닝 토크 특성 곡선 확인
- ✓ C_alpha, μ, trail 파라미터 변화에 따른 응답 비교
- ✓ Reset 기능 테스트


In [ ]:

print("=" * 60)
print("Lateral Tire Model Test Summary")
print("=" * 60)
print("✓ 기본 파라미터 및 초기 상태 확인 완료")
print("✓ 정속 조향 시나리오 테스트 완료")
print("✓ 슬립각-횡력/얼라이닝 토크 특성 검증 완료")
print("✓ C_alpha, μ, trail 파라미터 변화 테스트 완료")
print("✓ Reset 기능 테스트 완료")
print("모든 테스트 셀 실행을 완료했습니다!")
print("=" * 60)
